# Week 1, Assignment #1: Web Data Scraping

**Course:** Data & AI Specialist Program, Cyber Shujaa
**Opens:** Monday, 7 September 2026
**Due:** Sunday, 13 September 2026, 23:59
**Submit:** this notebook (.ipynb) plus your exported .csv file, via the file submission area on the LMS

## What this assignment is actually testing

In class we walked through the Data Science Methodology. This assignment is your first hands-on
pass through two of its steps: **Data Understanding** (going and getting real data instead of
being handed a clean file) and **Data Preparation** (fixing up what you collected before it is
usable). A live website is messier than any textbook dataset, which is exactly the point.

## Your target site (fixed, not a free choice)

Every student scrapes the same page for this assignment:
[scrapethissite.com/pages/simple/](https://www.scrapethissite.com/pages/simple/), "Countries of
the World: A Simple Example." It lists all 250 countries in the world, each with a capital,
population, and area. Using one fixed page keeps the assignment fair to grade and makes sure
everyone hits the same real-world snag: this page is **not** built as a `<table>`, unlike the
hockey site from the in-class demo, so you cannot reuse that code unchanged. You will apply the
same four-step pattern (fetch, parse, find the repeating structure, collect into a DataFrame) to
HTML shaped differently, which is the actual skill this assignment is testing.

## How this notebook works

This is a **scaffold, not a solution**. Some cells run as they are. Others have a `# TODO`
comment where you need to write real code yourself, using what you learned in the in-class demo
plus your own investigation of this page's HTML in DevTools. If a cell runs without errors but
you never replaced a `___` placeholder, you have not actually finished that step.

## Academic honesty

Everyone is scraping the same page, so it will be tempting to just copy a classmate's finished
selectors. Don't, the reflection questions at the end ask you to explain choices you made while
writing this code, and a script you did not actually write yourself will not let you answer them
honestly. Find the tags and classes below through your own inspection of the page.


## Step 1: Import your libraries

These three libraries do three different jobs in this pipeline: `requests` talks to the
website over HTTP, `BeautifulSoup` (from the `bs4` package) turns raw HTML text into something
you can search through, and `pandas` gives you the DataFrame you will store your results in.


In [1]:
# Colab already has these installed, but this line is harmless if they are already present
# and saves you a confusing NameError if you are running this outside Colab.
!pip install requests beautifulsoup4 pandas --quiet

import requests
from bs4 import BeautifulSoup
import pandas as pd


## Step 2: Inspect your target page

Your target is fixed: `https://www.scrapethissite.com/pages/simple/`. Before writing any
scraping code, check whether scraping is allowed: visit
`https://www.scrapethissite.com/robots.txt` in your browser and read it.

**Do this now, before running the next cell:**
1. Open the target page in a normal browser tab.
2. Right-click on one of the 250 country blocks and choose "Inspect" (or press F12).
3. Find the HTML tag and class name that wraps **one single country** (not the whole page, not
   just the country's name). You will need this exact tag and class name in Step 5.
4. While you are in there, also find the tags used for the capital, population, and area inside
   one country block. You will need those in Step 6.


In [2]:
url = "https://www.scrapethissite.com/pages/simple/"  # fixed target, same for everyone

# TODO: write one line below stating what you found at scrapethissite.com/robots.txt.
# Example: robots_txt_notes = "Disallow: /lessons/ and /faq/ only, /pages/ is not listed, safe to scrape."
robots_txt_notes = "The file disallows /lessons/ and /faq/ but does not list /pages/simple/, so scraping this page is permitted."

print(f"Target URL: {url}")
print(f"robots.txt check: {robots_txt_notes}")


Target URL: https://www.scrapethissite.com/pages/simple/
robots.txt check: The file disallows /lessons/ and /faq/ but does not list /pages/simple/, so scraping this page is permitted.


## Step 3: Fetch the page

`requests.get()` does not throw an error just because a page returns a 404 or a 500. It happily
hands you back a broken response and lets you find out the hard way three cells later when
`soup.find()` returns `None`. Catch that here instead.


In [3]:
response = requests.get(url)

# Check for errors immediately after the response, before printing anything,
# so a failed request stops here instead of printing misleading success output
response.raise_for_status()


print(f"Status code: {response.status_code}")
print(f"Bytes received: {len(response.text)}")

Status code: 200
Bytes received: 203338


## Step 4: Parse the HTML

`response.text` is one enormous string. `BeautifulSoup` turns it into a tree you can actually
search through by tag and class, instead of doing string matching yourself.


In [4]:
soup = BeautifulSoup(response.text, "html.parser")

# Print the first 800 characters so you can see the structure you are about to search through.
# This should look like real, readable HTML, not an error page or a login wall.
print(soup.prettify()[:800])


<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Countries of the World: A Simple Example | Scrape This Site | A public sandbox for learning web scraping
  </title>
  <link href="/static/images/scraper-icon.png" rel="icon" type="image/png"/>
  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>
  <meta content="A single page that lists information about all the countries in the world. Good for those just get started with web scraping." name="description"/>
  <link crossorigin="anonymous" href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" rel="stylesheet"/>



## Step 5: Find every repeating item on the page

This is the main challenge in this assignment. Every country on the page is wrapped in the same
repeating tag and class, the one you found with DevTools in Step 2. `soup.find_all()` returns
every element matching that tag and class as a list, all 250 of them.


In [5]:
# From inspection we found the tag to be div and class as country is the specific marker,
# unlike col-md-4 which is a generic Bootstrap layout class
items = soup.find_all("div", class_="country")

print(f"Found {len(items)} items on the page.")

# 250 items found which matched the page's stated count, confirming the selector is correct.


Found 250 items on the page.


## Step 6: Extract the fields you actually care about

For each country found in Step 5, pull out four fields: **name, capital, population, and
area**. This is where most of the real work in this assignment lives: the exact inner tags and
classes for each field are yours to find in DevTools, the same way you found the repeating
container in Step 2.


In [6]:
scraped_rows = []

# For each country block, search only inside that item so we get the specific data for that country,
# not a mismatched value from a different country.

for item in items:
# .strip() removes the whitespace left over from the HTML source, since without
# it comparing or matching this text elsewhere would fail even though it looks correct.
    row = {
        "name": item.find("h3", class_="country-name").text.strip(),
        "capital": item.find("span", class_="country-capital").text.strip(),
        "population": item.find("span", class_="country-population").text.strip(),
        "area_km2": item.find("span", class_="country-area").text.strip(),
    }
    scraped_rows.append(row)
    pass

# This check exists so you find out right here if Step 6 is still empty, instead of getting a
# confusing empty DataFrame three cells from now.
assert len(scraped_rows) > 0, "scraped_rows is still empty. Fill in the loop above before continuing."
print(f"Collected {len(scraped_rows)} rows.")


Collected 250 rows.


## Step 7: Build your DataFrame

You built a tiny version of this in class from a single value. Here you are doing the same
thing at real scale, from a whole list of dictionaries instead of one.


In [7]:
df = pd.DataFrame(scraped_rows)

print(df.shape)
df.head()


(250, 4)


,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


## Step 8: Clean your data (Data Preparation)

Real scraped data is rarely usable as-is. This is step three of the Data Science Methodology
from class, and it is not optional here. At minimum, check for and handle each of the
following. Not every issue will exist in your data, but you need to actually check, not assume.


In [8]:
# There is nothing to fix as 0 was returned for all the columns meaning no null values
print(df.isnull().sum())

# There were zero duplicates which makes sense because there are 250 distinct countries
print(df.duplicated().sum())

# Both columns started as object because .text always returns strings, even when the content looks numeric.
# Fixed by converting to int and float(because of the decimal points) respectively,
# The conversion only takes effect by reassigning the result back to the column
# otherwise the change doesn't stick, it just gets thrown away.
df['population'] = pd.to_numeric(df['population'])
df['area_km2'] = pd.to_numeric(df['area_km2']).astype(float)

print(df.dtypes)
df.head()


name          0
capital       0
population    0
area_km2      0
dtype: int64
0
name           object
capital        object
population      int64
area_km2      float64
dtype: object


,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


## Step 9: Export to CSV

This is the deliverable file you submit alongside this notebook.


In [9]:
output_filename = "WebScrapedCountries_Chelsea"

df.to_csv(output_filename, index=False)

# Prove the export actually worked by reading the file back in, rather than just trusting that
# to_csv() did not raise an error.
check_df = pd.read_csv(output_filename)
print(f"Saved and reloaded {check_df.shape[0]} rows, {check_df.shape[1]} columns.")
check_df.head()


Saved and reloaded 250 rows, 4 columns.


,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


## Step 10: Reflection (required, answer in this cell)

Replace the placeholders below with your own two or three sentence answers.

1. **What was the hardest part of getting your selectors right in Step 5 or Step 6, and how did
   you eventually figure it out?**
   *The hardest part was the two class element and having to reason out which specific class to filter on . And realized I should not filter on the bootstrap class but actually class name country.*

2. **What did Step 8 reveal about your data that you did not expect before you actually looked
   at it closely?**
   *It revealed to me that just changing the type is not enough I needed to reassign it back to that column for that type to actually reflect. This surprised me because the code ran without any error making it look like it work but on check the type it still remained object.*

3. **If you had another hour, what would you add or fix in this scraper?**
   *In Step 3 error-handling stops the whole notebook on a bad request instead I would use a try/except with retry l logic that maybe retries a couple of times incase the error was temporary instead of failing on the first try.*


## Submission checklist

- [ ] `robots_txt_notes` in Step 2 is filled in with your own real check, not a placeholder
- [ ] Step 3's status check actually stops on a failed request, not just prints a status code
- [ ] `items` in Step 5 is a non-empty list, roughly 250 items matching the page's own count
- [ ] `scraped_rows` in Step 6 has all four columns (name, capital, population, area) per row
- [ ] Step 8 shows evidence you actually checked for missing values, duplicates, and wrong types
- [ ] The exported .csv opens correctly when reloaded in Step 9
- [ ] All three reflection questions in Step 10 are answered in your own words
- [ ] You have submitted both this notebook (.ipynb) and your exported .csv file on the LMS

**Grading scale:** 0 = No submission, 1 = Below Expectation, 2 = Meets Expectation,
3 = Exceeds Expectation.